# 🔬 Science Paper Analyzer — полное руководство по кодуЭтот ноутбук подробно объясняет **каждый файл** проекта.  Код можно **не запускать** — просто читайте.**Быстрые ссылки:**  — Запустить анализ: [science_paper_analyzer.ipynb](https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/science_paper_analyzer.ipynb)  — GitHub: https://github.com/DmitPerson42/science-paper-analyzer

## 📑 Содержание| № | Файл | Описание ||---|------|----------|| 1 | `science_paper_analyzer.ipynb` | Главный Colab — 3 шага || 2 | `parsers/utils.py` | `extract_year()` — конвертация года || 3 | `parsers/arxiv_parser.py` | Парсер arXiv.org || 4 | `parsers/semantic_scholar.py` | Парсер Semantic Scholar || 5 | `parsers/openreview_parser.py` | Парсер OpenReview.net || 6 | `parsers/aclanthology.py` | Парсер ACL Anthology || 7 | `parsers/jmlr_parser.py` | Парсер JMLR.org || 8 | `parsers/crossref_fallback.py` | Парсер через CrossRef || 9 | `parsers/cyberleninka.py` | Парсер CyberLeninka.ru || 10 | `parsers/utils.py` (ещё раз) | Вспомогательные функции || 11 | `analyzers/citation_analyzer.py` | Анализ цитируемости || 12 | `analyzers/text_analyzer.py` | AI-детектор текста || 13 | `analyzers/journal_analyzer.py` | Beall's List || 14 | `exporters/csv_exporter.py` | Экспорт в CSV || 15 | `exporters/excel_exporter.py` | Экспорт в Excel || 16 | `analyzer.py` | Центральный координатор || 17 | `app.py` | Streamlit веб-интерфейс || 18 | `requirements.txt` | Зависимости |

---## 1. science_paper_analyzer.ipynb — главный ColabТри шага: **установка → анализ → скачать**.### Шаг 1: Клонирование кода и установка

In [ ]:
import subprocess, sys, os, importlib# Откуда скачать кодREPO_URL = "https://github.com/DmitPerson42/science-paper-analyzer.git"REPO_DIR = "science-paper-analyzer"# Если папка уже есть — удаляем (чтобы всегда свежая версия)if os.path.exists(REPO_DIR):    import shutil    shutil.rmtree(REPO_DIR, ignore_errors=True)# Клонируем репозиторийsubprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True,               capture_output=True, text=True)os.chdir(REPO_DIR)# Проверяем, какие пакеты уже установленыrequired = ["requests", "pandas", "openpyxl", "lxml",            "beautifulsoup4", "numpy", "scikit-learn"]missing = [p for p in required if importlib.util.find_spec(p) is None]# Устанавливаем только то, чего нетif missing:    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing,                   check=True, capture_output=True, text=True)print("✅ Шаг 1 выполнен! Переходите к Шагу 2.")

**Что здесь происходит:**1. `subprocess` — запускает внешние команды (`git clone`, `pip install`).2. `shutil.rmtree()` — удаляет старую версию кода. Так вы всегда работаете с последней.3. `git clone --depth 1` — скачивает только последний коммит (быстро, без истории).4. `importlib.util.find_spec()` — проверяет, импортируется ли библиотека. Если нет — она в списке `missing`.5. `pip install -q` — тихая установка без лишнего вывода.---### Шаг 2: Запуск анализа

In [ ]:
import sys, pandas as pdsys.path.insert(0, ".")from analyzer import PaperAnalyzer# ------ НАСТРОЙКА ------QUERY = "filtering generative text data language model collapse"MAX_PER_SOURCE = 5# -----------------------print(f"📌 Тема: {QUERY}")print(f"📌 Статей/источник: {MAX_PER_SOURCE}\n")# Создаём главный объект программыanalyzer = PaperAnalyzer(verbose=True)# --- СБОР СТАТЕЙ С 10 ИСТОЧНИКОВ ---print("="*50)print("📡 СБОР СТАТЕЙ")print("="*50)papers = analyzer.collect_papers(QUERY, max_per_source=MAX_PER_SOURCE)if not papers:    print("\n❌ Статей не найдено. Попробуйте другую тему.")    df = pd.DataFrame(columns=["title","authors","year","source",                               "url","abstract","citation_score",                               "text_score","journal_score",                               "overall_score","verdict"])else:    # --- АНАЛИЗ ПО ТРЁМ КРИТЕРИЯМ ---    print("\n" + "="*50)    print("🔬 АНАЛИЗ СТАТЕЙ")    print("="*50)    df = analyzer.analyze_papers(papers)    print(f"\n✅ Проанализировано {len(df)} статей.\n")    # --- ТАБЛИЦА ---    display_cols = ["title", "authors", "year", "source",                    "overall_score", "verdict"]    df_view = df[display_cols].copy()    df_view.columns = ["📖 Название", "👤 Авторы", "📅 Год",                       "📡 Источник", "Score", "⚖️ Вердикт"]    display(df_view)print("\n✅ Анализ завершён!")

**Что здесь происходит:**1. `sys.path.insert(0, ".")` — добавляем текущую папку в пути поиска Python.     Без этого `from analyzer import PaperAnalyzer` не сработает.2. `PaperAnalyzer(verbose=True)` — создаём экземпляр главного класса.     `verbose=True` включает вывод в консоль (для Colab).3. `collect_papers()` — проходит по 10 источникам, собирает статьи в список словарей.4. `analyze_papers()` — прогоняет каждую статью через 3 анализатора, возвращает DataFrame.5. `display(df_view)` — Colab показывает красивую таблицу.---### Шаг 3: Скачать результаты

In [ ]:
if "df" in dir() and len(df) > 0:    csv_path = "analysis_results.csv"    xlsx_path = "analysis_results.xlsx"    analyzer.export_csv(df, csv_path)    analyzer.export_excel(df, xlsx_path)    from google.colab import files    files.download(csv_path)    print("✅ CSV скачан!")else:    print("⚠️ Нет данных для экспорта. Сначала выполните Шаг 2.")

**Проверка:** `"df" in dir()` — есть ли переменная `df` в памяти.  Если анализ не запускали — переменной нет, и экспорт не упадёт с ошибкой.---## 2. parsers/utils.py — extract_year()Самая частая причина багов: источники возвращают год в разных форматах.

In [ ]:
import refrom datetime import datetime, timezonedef extract_year(value) -> str:    # Пустые значения — сразу выход    if value is None or value == "":        return ""    # Если число    if isinstance(value, (int, float)):        # Если похоже на год (1900-2050)        if 1900 <= value <= 2050:            return str(int(value))        # Если большое число — это timestamp (дата в цифрах)        if value > 10**9:            if value > 10**12:                value = value / 1000  # миллисекунды → секунды            try:                return str(datetime.fromtimestamp(value, tz=timezone.utc).year)            except:                return ""        return ""    # Если строка — ищем 4 цифры (год)    s = str(value).strip()    match = re.search(r'\b(19[0-9][0-9]|20[0-9][0-9])\b', s)    if match:        return match.group(1)    return ""

**Зачем нужна эта функция?**  OpenReview возвращает дату как `1652000000000` (миллисекунды с 1970 года).  Если взять первые 4 символа: `str(1652000000000)[:4]` → **"1652"**.  Функция `extract_year()` понимает, что это timestamp, конвертирует: **"2022"**.**Поддерживаемые форматы:**- Число `2023` → `"2023"`- Строка `"2023-01-15"` → `"2023"`- Строка `"2023 год"` → `"2023"`- Timestamp `1672531200` (секунды) → `"2023"`- Timestamp `1672531200000` (миллисекунды) → `"2023"`- None / пусто → `""` (не падает)---## 3. parsers/arxiv_parser.py — парсер arXivarXiv — бесплатный архив научных статей. Есть REST API.

In [ ]:
import requests, timeimport xml.etree.ElementTree as ETfrom typing import List, Dictfrom parsers.utils import extract_yearARXIV_API = "https://export.arxiv.org/api/query"class ArxivParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {            "search_query": f"all:{query}",            "start": 0,            "max_results": max_results        }        headers = {"User-Agent": "SciencePaperAnalyzer/1.0"}        # Повторные попытки при ошибках (до 3 раз)        max_retries = 3        for attempt in range(max_retries):            try:                resp = requests.get(ARXIV_API, params=params, headers=headers, timeout=30)                if resp.status_code == 429:  # "Too Many Requests"                    time.sleep((attempt + 1) * 3)                    continue                resp.raise_for_status()                break            except requests.exceptions.HTTPError:                if attempt == max_retries - 1:                    return []  # тихо возвращаем пустой список                time.sleep(2)        time.sleep(1)  # вежливая пауза        # arXiv возвращает XML, парсим его        root = ET.fromstring(resp.text)        ns = {"a": "http://www.w3.org/2005/Atom"}        papers = []        for entry in root.findall("a:entry", ns):            title = entry.findtext("a:title", "").strip().replace("\n", " ")            summary = entry.findtext("a:summary", "").strip().replace("\n", " ")            published = entry.findtext("a:published", "")            link_el = entry.find("a:id", ns)            link = link_el.text if link_el is not None else ""            authors = []            for author in entry.findall("a:author", ns):                name = author.findtext("a:name", "")                if name:                    authors.append(name)            year = extract_year(published)            papers.append({                "id": link.split("/")[-1],      # arXiv ID (например, 2301.12345)                "title": title,                "authors": ", ".join(authors[:5]),  # первые 5 авторов                "year": year,                "abstract": summary,                "url": link,                "venue": "arXiv",            })        return papers

**Детали:**1. `"all:..."` — искать во всех полях (название, аннотация, авторы).2. `resp.status_code == 429` — сервер говорит "подожди". Ждём 3, 6, 9 секунд.3. `ET.fromstring(resp.text)` — arXiv возвращает XML в формате Atom. `ns` — пространство имён.4. `entry.findtext("a:title", "")` — ищем текст внутри тега `<title>`. Если нет — возвращаем `""`.5. `authors[:5]` — ограничиваем до 5, чтобы таблица не раздувалась.---## 4. parsers/semantic_scholar.py — парсер Semantic ScholarЕдинственный бесплатный источник, дающий **количество цитирований**.

In [ ]:
import requestsfrom typing import List, DictSEMANTIC_API = "https://api.semanticscholar.org/graph/v1/paper/search"FIELDS = "title,authors,year,citationCount,abstract,venue,externalIds,url"class SemanticScholarParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"query": query, "limit": max_results, "fields": FIELDS}        headers = {"User-Agent": "SciencePaperAnalyzer/1.0"}        resp = requests.get(SEMANTIC_API, params=params, headers=headers, timeout=30)                if resp.status_code == 429:            raise Exception("Rate limited. Please wait and try again.")        resp.raise_for_status()        data = resp.json()        papers = []        for paper in data.get("data", []):            authors = [a.get("name", "") for a in paper.get("authors", [])]            ext_ids = paper.get("externalIds", {}) or {}  # может быть null            papers.append({                "id": paper.get("paperId", ""),                "title": paper.get("title", "N/A"),                "authors": ", ".join(authors[:5]),                "year": str(paper.get("year", "")) if paper.get("year") else "",                "abstract": paper.get("abstract", "") or "",                "url": f"https://www.semanticscholar.org/paper/{paper.get('paperId', '')}",                "citation_count": paper.get("citationCount", 0),  # <-- ключевое поле                "venue": paper.get("venue", ""),                "doi": ext_ids.get("DOI", ""),            })        return papers

**Важное поле:** `citationCount` — сколько раз статью процитировали.  Без этого поля анализатор цитируемости не сможет работать.**Проблема:** Бесплатный API имеет лимит ~2 запроса в секунду.  На практике в Colab часто получает 429 (Too Many Requests).**`ext_ids = paper.get("externalIds", {}) or {}`** — защита от `null`.  API может вернуть `"externalIds": null`, и код упадёт. `or {}` превращает null в `{}`.---## 5. parsers/openreview_parser.py — парсер OpenReviewOpenReview — платформа для рецензирования конференций (ICLR, NeurIPS и др.).

In [ ]:
import requestsfrom typing import List, Dictfrom parsers.utils import extract_yearOPENREVIEW_API = "https://api.openreview.net/notes/search"class OpenReviewParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"term": query, "limit": max_results, "source": "forum"}        headers = {"User-Agent": "SciencePaperAnalyzer/1.0"}        # Пробуем основной эндпоинт        try:            resp = requests.get(OPENREVIEW_API, params=params, headers=headers, timeout=30)            resp.raise_for_status()        except Exception:            # Fallback: другой эндпоинт            alt_url = f"https://api.openreview.net/notes?term={query}&limit={max_results}"            resp = requests.get(alt_url, headers=headers, timeout=30)            resp.raise_for_status()        data = resp.json()        papers = []        # OpenReview может вернуть "notes" или "rows"        notes = data.get("notes", [])        if not notes:            notes = data.get("rows", [])        for note in notes[:max_results]:            content = note.get("content", {})                        # OpenReview оборачивает поля в {"value": "..."}            title = content.get("title", note.get("title", "N/A"))            if isinstance(title, dict):                title = title.get("value", "N/A")            authors = content.get("authors", note.get("authors", []))            if authors and isinstance(authors, dict):                authors = authors.get("value", [])            abstract = content.get("abstract", "")            if isinstance(abstract, dict):                abstract = abstract.get("value", "")            forum = note.get("forum", "")            cdate = note.get("cdate", note.get("tcdate", 0))            year = extract_year(cdate)  # !!! ЗДЕСЬ БЫЛ БАГ !!!            papers.append({                "id": note.get("id", ""),                "title": title if isinstance(title, str) else str(title),                "authors": ", ".join(authors[:5]) if isinstance(authors, list) else str(authors),                "year": year,                "abstract": abstract,                "url": f"https://openreview.net/forum?id={forum}" if forum else "",                "venue": "OpenReview",            })        return papers

**❗ БАГ, КОТОРЫЙ МЫ ИСПРАВИЛИ**Старая версия: `year = str(cdate)[:4]`  `cdate` — это **миллисекунды с 1970 года**. Например, `1652000000000`.  Первые 4 символа: **"1652"** — бредовый год.Новая версия: `year = extract_year(cdate)`  Функция понимает, что это timestamp, и возвращает **"2022"**.**Другие особенности OpenReview:**1. Метаданные лежат внутри `content` и обёрнуты в `{"value": "..."}` — нужна проверка `isinstance(...)`.2. Есть два разных эндпоинта — `/notes/search` и `/notes`.3. `tcdate` — время создания темы, `cdate` — время создания заметки.---## 6. parsers/aclanthology.py — парсер ACL AnthologyACL Anthology — архив публикаций по компьютерной лингвистике.

In [ ]:
import requestsfrom typing import List, DictACL_API = "https://api.aclanthology.org/v1/search"class ACLAnthologyParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {"q": query, "limit": max_results}        headers = {"User-Agent": "SciencePaperAnalyzer/1.0"}        try:            resp = requests.get(ACL_API, params=params, headers=headers, timeout=30)            resp.raise_for_status()        except Exception:            return self._scrape_search(query, max_results)  # fallback        data = resp.json()        papers = []        results = data.get("results", data.get("data", []))        for paper in results[:max_results]:            anth_id = paper.get("paper_id", paper.get("id", ""))            title = paper.get("title", "N/A")            authors_list = paper.get("authors", [])            year = str(paper.get("year", ""))            abstract = paper.get("abstract", paper.get("paper_abstract", ""))            papers.append({                "id": anth_id,                "title": title,                "authors": ", ".join(authors_list[:5]) if isinstance(authors_list, list) else str(authors_list),                "year": year,                "abstract": abstract,                "url": f"https://aclanthology.org/{anth_id}/" if anth_id else "",                "venue": "ACL Anthology",            })        return papers    def _scrape_search(self, query, max_results):        # Fallback на HTML scraping, если API не отвечает        from bs4 import BeautifulSoup        url = f"https://aclanthology.org/?q={query}"        resp = requests.get(url, headers=headers, timeout=30)        soup = BeautifulSoup(resp.text, "lxml")        # ... парсинг HTML        return papers

**Детали:**1. `results = data.get("results", data.get("data", []))` — разные версии API.2. `_scrape_search()` — запасной вариант через парсинг HTML-страницы.3. ID статьи вида `2023.acl-long.1` → URL `https://aclanthology.org/2023.acl-long.1/`.4. Год приходит как число → `str(paper.get("year", ""))`.---## 7. parsers/jmlr_parser.py — парсер JMLRJMLR (Journal of Machine Learning Research) — нет API, только HTML.

In [ ]:
import requests, refrom bs4 import BeautifulSoupfrom typing import List, DictJMLR_URL = "https://jmlr.org/papers/"def _jmlr_volume_to_year(href: str) -> str:    # URL: https://jmlr.org/papers/v22/i1.html    match = re.search(r'/v(\d+)', href)    if not match:        return ""    try:        vol = int(match.group(1))        # v1 = 2000, v2 = 2001, ..., v22 = 2021        year = 2000 + vol - 1        if 1999 <= year <= 2030:            return str(year)    except:        pass    return ""class JMLRParser:    def search(self, query, max_results=5):        resp = requests.get(JMLR_URL, ...)        soup = BeautifulSoup(resp.text, "lxml")        links = soup.select("a[href*='papers/v']")        # ...        for link in links:            href = link.get("href", "")            title = link.get_text(strip=True)            if query.lower() not in title.lower():                continue            papers.append({                "year": _jmlr_volume_to_year(href),                ...            })        return papers

**Как определяется год:**  У JMLR нет поля "year" в HTML. Номер тома = год.  `v1` = 2000, `v2` = 2001, ... `v22` = 2021.  Формула: `year = 2000 + volume - 1`.---## 8. parsers/crossref_fallback.py — через CrossRefCrossRef — центральный реестр публикаций. Используется для сайтов без API:  ResearchGate, eLibrary, Dissercat, FIPS.

In [ ]:
import requestsfrom typing import List, DictCROSSREF_API = "https://api.crossref.org/works"class CrossRefFallback:    def __init__(self, source_name: str = "CrossRef"):        # source_name = "ResearchGate", "eLibrary", "Dissercat" или "FIPS"        self.source_name = source_name    def search(self, query: str, max_results: int = 5) -> List[Dict]:        params = {            "query": query,            "rows": max_results,            "select": "title,author,DOI,URL,abstract,publication-date,publisher,container-title"        }        headers = {"User-Agent": "SciencePaperAnalyzer/1.0 (mailto:example@example.com)"}        try:            resp = requests.get(CROSSREF_API, params=params, headers=headers, timeout=30)            resp.raise_for_status()        except Exception as e:            return self._empty_result(query, str(e))        data = resp.json()        items = data.get("message", {}).get("items", [])        papers = []        for item in items[:max_results]:            title = item.get("title", ["N/A"])[0]  # title возвращается как список            authors_list = item.get("author", [])            authors = ", ".join([                f"{a.get('given', '')} {a.get('family', '')}".strip()                for a in authors_list[:5]            ])            doi = item.get("DOI", "")                        # Извлекаем год            date_parts = (item.get("published-online", {}).get("date-parts") or                         item.get("published-print", {}).get("date-parts") or                         item.get("issued", {}).get("date-parts") or [])            year = str(date_parts[0][0]) if date_parts and date_parts[0] else ""            papers.append({                "id": doi, "title": title, "authors": authors,                "year": year, "doi": doi,                "source_context": self.source_name,  # метка источника            })        if not papers:            return self._empty_result(query, "No results from CrossRef")        return papers

**Важно:** Создаётся **4 экземпляра** этого класса — по одному на каждый источник:  ```python"researchgate": CrossRefFallback(source_name="ResearchGate"),"elibrary":     CrossRefFallback(source_name="eLibrary"),"dissercat":    CrossRefFallback(source_name="Dissercat"),"fips":         CrossRefFallback(source_name="FIPS"),````source_name` просто подписывает результаты, чтобы пользователь знал, откуда они.**Особенности CrossRef:**1. `item.get("title", ["N/A"])[0]` — заголовок приходит как список `["Title"]`.2. Дата в формате `[[2023, 1, 15]]` — массив массивов.3. `"select": "..."` — ограничиваем поля, чтобы ответ был быстрее.---## 9. parsers/cyberleninka.py — парсер CyberLeninkaРоссийская научная библиотека. Нет API — только HTML.

In [ ]:
import requestsfrom bs4 import BeautifulSoupfrom typing import List, Dictfrom parsers.utils import extract_yearCYBERLENINKA_BASE = "https://cyberleninka.ru"class CyberLeninkaParser:    def search(self, query: str, max_results: int = 5) -> List[Dict]:        headers = {            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"        }        try:            url = f"{CYBERLENINKA_BASE}/search?q={query}&page=1"            resp = requests.get(url, headers=headers, timeout=30)            resp.raise_for_status()        except Exception as e:            return self._fallback_empty(query, str(e))        soup = BeautifulSoup(resp.text, "lxml")        papers = []        # Несколько CSS-селекторов — сайт меняет вёрстку        articles = soup.select("article, .search-item, .article-item, .b-serp-item")        if not articles:            articles = soup.select("li.result-item, div.result, .b-article")        for article in articles[:max_results]:            title_el = article.select_one("a[href*='/article/'], a[href*='article']")            if not title_el:                title_el = article.select_one("h2 a, h3 a, .title a")            href = title_el.get("href", "") if title_el else ""            if href and not href.startswith("http"):                href = CYBERLENINKA_BASE + href  # относительная → абсолютная            title = title_el.get_text(strip=True) if title_el else "N/A"                        year_el = article.select_one(".year, .date, .b-serp-item__date")            year = year_el.get_text(strip=True) if year_el else ""            papers.append({                "id": href.split("/")[-1] if href else "",                "title": title,                "year": extract_year(year),  # "2023 год" → "2023"                ...            })        if not papers:            return self._fallback_empty(query, "Empty results")        return papers

**Проблемы парсинга CyberLeninka:**1. User-Agent должен быть как у браузера, иначе блокируют.2. Вёрстка меняется — приходится перебирать несколько CSS-селекторов.3. Ссылки могут быть относительными (`/article/12345`) — превращаем в абсолютные.4. `extract_year(year)` — на случай, если год написан как "2023 год".---## 10. analyzers/citation_analyzer.py — анализ цитируемостиПроверяет, сколько раз статью процитировали.

In [ ]:
from typing import Tupleimport datetimeclass CitationAnalyzer:    def analyze(self, paper: dict) -> Tuple[float, str]:        # Ищем citation_count в любом поле        citation_count = paper.get("citation_count", None)        if citation_count is None:            citation_count = paper.get("citations", None)        if citation_count is None:            citation_count = paper.get("cited_by_count", None)        # Парсим год        year_str = paper.get("year", "")        try:            year = int(year_str) if year_str and year_str.isdigit() else None        except:            year = None        # Нет данных — нейтрально        if citation_count is None:            return 0.5, "Нет данных о цитированиях — нейтрально"        citation_count = int(citation_count)        current_year = datetime.datetime.now().year        # Логика:        if year and (current_year - year) >= 3 and citation_count == 0:            # Статья старше 3 лет, ни одного цитирования = подозрительно            return 0.2, f"Статья {year} года, {citation_count} цит. — подозрительно"        elif citation_count == 0:            return 0.5, f"{citation_count} цит. — нейтрально (молодая статья)"        elif citation_count <= 5:            return 0.7, f"{citation_count} цит. — приемлемый уровень"        elif citation_count <= 15:            return 0.85, f"{citation_count} цит. — хороший уровень"        else:            return 0.95, f"{citation_count} цит. — высокий уровень доверия"

**Логика:**| Ситуация | Score | Почему ||----------|-------|--------|| Статья 2020 года, 0 цит. | **0.2** | За 3+ года хорошую статью хоть раз процитируют || Статья 2024 года, 0 цит. | **0.5** | Новая, ещё не успели || 1-5 цитирований | **0.7** | Нормально || 5-15 цитирований | **0.85** | Хорошо || 15+ цитирований | **0.95** | Высокое доверие |---## 11. analyzers/text_analyzer.py — AI-детектор текстаЭвристический детектор (без нейросетей). Использует 5 метрик.

In [ ]:
from typing import Tupleimport reclass TextAnalyzer:    def __init__(self):        # Фразы, которые часто используют AI-модели        self.ai_markers = [            "as an ai", "as a language model", "i cannot",            "in conclusion", "additionally", "furthermore",            "in recent years", "state-of-the-art", "cutting-edge",            "groundbreaking", "leveraging", "harnessing the power",        ]    def analyze(self, paper: dict) -> Tuple[float, str]:        # Собираем весь текст (название + аннотация)        text = ""        for field in ["title", "abstract", "summary"]:            val = paper.get(field, "")            if val:                text += " " + str(val)        text = text.strip()        if len(text) < 50:            return 0.5, "Недостаточно текста для анализа"        words = text.split()        text_lower = text.lower()        n_words = len(words)        # Метрика 1: AI-маркеры        marker_hits = sum(1 for m in self.ai_markers if m in text_lower)        marker_score = 1.0 - min(marker_hits / len(self.ai_markers) * 3, 0.8)        # Метрика 2: Лексическое разнообразие (TTR)        unique_words = set(w.lower() for w in words)        ttr = len(unique_words) / n_words        ttr_score = 1.0 - abs(ttr - 0.65) / 0.65  # идеал = 0.65        # Метрика 3: Повторяющиеся 3-граммы        repeat_penalty = 0.0        if n_words > 10:            seen = set()            for i in range(n_words - 2):                ngram = " ".join(words[i:i+3]).lower()                if ngram in seen:                    repeat_penalty += 0.02                seen.add(ngram)            repeat_penalty = min(repeat_penalty, 0.5)        # Метрика 4: Равномерность длины предложений        sentences = re.split(r'[.!?]+', text)        sent_lengths = [len(s.split()) for s in sentences if len(s.split()) > 2]        if sent_lengths:            avg = sum(sent_lengths) / len(sent_lengths)            std = (sum((l - avg)**2 for l in sent_lengths) / len(sent_lengths)) ** 0.5            rel_std = std / max(avg, 1)            if rel_std < 0.3:                uniformity_penalty = 0.2  # слишком ровно → AI            elif rel_std > 1.2:                uniformity_penalty = 0.1            else:                uniformity_penalty = 0.0        else:            uniformity_penalty = 0.0        # Метрика 5: Доля стоп-слов        stop_words = {"the", "a", "an", "and", "or", "in", "on", "at",                      "of", "to", "for", "with", "by", "is", "are", "this"}        stop_ratio = sum(1 for w in words if w.lower() in stop_words) / n_words        stop_penalty = min((stop_ratio - 0.60) * 2, 0.3) if stop_ratio > 0.60 else 0.0        # Финальный score (взвешенная сумма)        score = (marker_score * 0.3 +                 ttr_score * 0.2 +                 (1.0 - repeat_penalty) * 0.25 +                 (1.0 - uniformity_penalty) * 0.1 +                 (1.0 - stop_penalty) * 0.15)        score = max(0.0, min(1.0, score))        return score, detail_message

**5 метрик и почему они работают:**1. **AI-маркеры:** ChatGPT часто использует шаблонные фразы (`"as an AI"`, `"in conclusion"`). Чем их больше — тем вероятнее AI.2. **TTR (Type-Token Ratio):** отношение уникальных слов к общему количеству. У людей словарь разнообразнее (TTR ~0.65), у AI — беднее.3. **Повторяющиеся 3-граммы:** AI часто повторяет одинаковые тройки слов. Люди — реже.4. **Равномерность длины предложений:** AI пишет предложения примерно одинаковой длины. Люди чередуют короткие и длинные.5. **Стоп-слова:** AI использует больше служебных слов (the, a, of). У людей — меньше.Каждый score от 0 до 1, итоговый — взвешенная сумма весов.---## 12. analyzers/journal_analyzer.py — проверка журнала (Beall's List)Beall's List — список "хищнических" журналов, которые публикуют статьи за деньги без рецензирования.

In [ ]:
from typing import Tuple# Хищнические журналы (сокращённый Beall's List, ~180)PREDATORY_JOURNALS = [    "international journal of engineering research and technology",    "journal of emerging technologies and innovative research",    "waset (world academy of science, engineering and technology)",    "science publishing group",    # ... всего ~180 наименований]# Доверенные площадкиTRUSTED_VENUES = [    "nature", "science", "ieee", "acl", "neurips", "icml",    "arxiv", "jmlr", "springer", "elsevier", "acm",    "openreview", "semantic scholar",    # ... ~50 наименований]class JournalAnalyzer:    def __init__(self):        self.predatory_set = set(PREDATORY_JOURNALS)  # set = быстрый поиск        self.trusted_set = set(v.lower() for v in TRUSTED_VENUES)    def analyze(self, paper: dict) -> Tuple[float, str]:        # Собираем все поля с названием площадки        venue_text = " ".join([            str(paper.get("venue", "")),            str(paper.get("source", "")),            str(paper.get("publisher", "")),        ]).lower().strip()        if not venue_text:            return 0.5, "Нет данных о журнале"        # Проверка доверенных        for trusted in self.trusted_set:            if trusted in venue_text:                return 0.95, f"Площадка '{paper.get('venue','')}' доверенная"        # Проверка хищнических        for predatory in self.predatory_set:            if predatory in venue_text:                return 0.15, f"Журнал в Beall's List (хищнический)"        return 0.6, "Журнал не найден в списках — нейтрально"

**Логика:**1. Собираем название площадки из нескольких полей (venue, source, publisher).2. Если площадка в `TRUSTED_VENUES` — score **0.95** (Nature, Science, IEEE, arXiv...).3. Если площадка в `PREDATORY_JOURNALS` — score **0.15** (хищнический журнал).4. Если не нашли ни там, ни там — **0.6** (нейтрально).**Важно:** `trusted in venue_text` — частичное совпадение.  Если article написано в "Nature Communications", а в списке есть "nature" — совпадение найдётся.---## 13. exporters/csv_exporter.py — экспорт в CSV

In [ ]:
import pandas as pdclass CSVExporter:    def export(self, df: pd.DataFrame, filepath: str):        # Основные колонки (без деталей)        export_cols = [            "title", "authors", "year", "source", "url",            "citation_score", "text_score", "journal_score",            "overall_score", "verdict"        ]        # Защита: оставляем только те колонки, что есть        export_cols = [c for c in export_cols if c in df.columns]        df_export = df[export_cols].copy()        df_export.to_csv(filepath, index=False, encoding="utf-8-sig")        return filepath

`encoding="utf-8-sig"` — добавляет BOM в начало файла.  Это нужно, чтобы **Excel на Windows** правильно открыл русские буквы.---## 14. exporters/excel_exporter.py — экспорт в Excel (с цветами)

In [ ]:
from openpyxl import Workbookfrom openpyxl.styles import PatternFill, Font, Alignment, Border, Sidefrom openpyxl.utils.dataframe import dataframe_to_rowsclass ExcelExporter:    def export(self, df: pd.DataFrame, filepath: str):        wb = Workbook()        ws = wb.active        ws.title = "Results"        # Все колонки (включая детали-пояснения)        export_cols = [            "title", "authors", "year", "source", "url",            "citation_score", "text_score", "journal_score",            "overall_score", "verdict",            "citation_detail", "text_detail", "journal_detail",        ]        export_cols = [c for c in export_cols if c in df.columns]        df_export = df[export_cols].copy()        # Запись данных        for row in dataframe_to_rows(df_export, index=False, header=True):            ws.append(list(row))        # Стиль заголовка: тёмно-синий фон, белый текст        header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")        header_font = Font(color="FFFFFF", bold=True, size=11)        for cell in ws[1]:            cell.fill = header_fill            cell.font = header_font        # Цвета для вердикта        green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")        yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")        red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")        # Красим строки по вердикту        for row in ws.iter_rows(min_row=2, max_col=1, max_row=ws.max_row):            cell = row[0]  # первая ячейка строки            if cell.value:                val = str(cell.value)                if "Real" in val:                    cell.fill = green_fill                elif "Suspicious" in val:                    cell.fill = yellow_fill                elif "Fake" in val:                    cell.fill = red_fill        # Ширина колонок        ws.column_dimensions["A"].width = 60   # title        ws.column_dimensions["B"].width = 40   # authors        ws.column_dimensions["C"].width = 10   # year        ws.column_dimensions["D"].width = 20   # source        ws.column_dimensions["E"].width = 40   # url        wb.save(filepath)        return filepath

**Цвета:**- Зелёный (`#C6EFCE`) — Real ✅- Жёлтый (`#FFEB9C`) — Suspicious ⚠️- Красный (`#FFC7CE`) — Fake ❌Детали с пояснениями (`citation_detail`, `text_detail`, `journal_detail`)  тоже сохраняются в Excel, чтобы можно было понять, почему такой вердикт.---## 15. analyzer.py — центральный координаторСамый важный файл. Собирает всё вместе.### Заглушка для Streamlit

In [ ]:
try:    import streamlit as st    _has_streamlit = Trueexcept ImportError:    # В Colab нет Streamlit — создаём заглушку    class _Stub:        def progress(self, *a, **kw): return self        def empty(self, *a, **kw): return self        def warning(self, msg): print(f"[WARNING] {msg}")        def __getattr__(self, name): return lambda *a, **kw: None    st = _Stub()    _has_streamlit = False

**Зачем это нужно?**  В `analyzer.py` есть вызовы `st.progress(0)`, `st.warning(...)` — для веб-интерфейса.  Если Streamlit не установлен (как в Colab), эти вызовы упадут с ошибкой.  Заглушка `_Stub` подменяет все методы Streamlit на функции, которые ничего не делают.  Теперь код работает и в Colab, и в вебе.### PaperAnalyzer.collect_papers()

In [ ]:
def collect_papers(self, query: str, max_per_source: int = 5):    all_papers = []    for idx, (source_key, parser) in enumerate(self.parsers.items()):        try:            papers = parser.search(query, max_results=max_per_source)            for p in papers:                p["source"] = source_key                p["source_display"] = SOURCE_DISPLAY.get(source_key, source_key)                if "id" not in p or not p["id"]:                    p["id"] = f"{source_key}_{hash(p.get('title', '')) % 100000}"            all_papers.extend(papers)            time.sleep(0.3)  # пауза между источниками        except Exception as e:            self._log(f"[WARNING] {source_key}: {str(e)[:100]}")    # Дедупликация    seen = set()    unique = []    for p in all_papers:        key = p.get("title", "").strip().lower()[:100]        if key and key not in seen:            seen.add(key)            unique.append(p)    return unique

**Что здесь происходит:**1. Проходит по всем 10 парсерам (arxiv, semantic_scholar, ...).2. Каждый парсер возвращает список статей.3. К каждой статье добавляется метка `source` — откуда пришла.4. Если у статьи нет ID — генерируется свой.5. `time.sleep(0.3)` — пауза между запросами, чтобы не забанили.6. **Дедупликация:** одинаковые названия статей удаляются.     Ключ = первые 100 символов названия в нижнем регистре.### PaperAnalyzer.analyze_papers()

In [ ]:
def analyze_papers(self, papers: list) -> pd.DataFrame:    results = []    for paper in papers:        # 3 анализатора        citation_score, citation_detail = self.analyzers["citation"].analyze(paper)        text_score, text_detail = self.analyzers["text"].analyze(paper)        journal_score, journal_detail = self.analyzers["journal"].analyze(paper)        scores = [citation_score, text_score, journal_score]        avg_score = sum(scores) / len(scores)  # среднее арифметическое        # Вердикт        if avg_score >= 0.7:            verdict = "Real"        elif avg_score >= 0.4:            verdict = "Suspicious"        else:            verdict = "Fake"        results.append({            "title": paper.get("title", "N/A"),            "authors": paper.get("authors", "N/A"),            "year": paper.get("year", "N/A"),            "citation_score": round(citation_score, 2),            "text_score": round(text_score, 2),            "journal_score": round(journal_score, 2),            "overall_score": round(avg_score, 2),            "verdict": verdict,            "citation_detail": citation_detail,            "text_detail": text_detail,            "journal_detail": journal_detail,        })    return pd.DataFrame(results)

**Формула:** `verdict_score = (citation_score + text_score + journal_score) / 3`| Средний score | Вердикт ||---------------|---------|| ≥ 0.70 | Real ✅ || 0.40 - 0.70 | Suspicious ⚠️ || < 0.40 | Fake ❌ |---## 16. app.py — веб-интерфейс (Streamlit)

In [ ]:
import streamlit as stfrom analyzer import PaperAnalyzer, SOURCE_DISPLAYst.set_page_config(page_title="Science Paper Analyzer", layout="wide")# Боковая панель с настройкамиwith st.sidebar:    query = st.text_input("Тема поиска:")    max_sources = st.slider("Источники", 1, 10, 10)    max_per_source = st.slider("Статей/источник", 1, 10, 5)        for key in all_sources[:max_sources]:        st.checkbox(SOURCE_DISPLAY[key], value=True)    if st.button("🚀 Запустить анализ"):        analyzer = PaperAnalyzer()        papers = analyzer.collect_papers(query, max_per_source)        df = analyzer.analyze_papers(papers)        st.session_state.results_df = df# Основное содержимоеif st.session_state.results_df is not None:    df = st.session_state.results_df        # Карточки статистики    col1, col2, col3 = st.columns(3)    col1.metric("Достоверные", real_count)    col2.metric("Подозрительные", sus_count)    col3.metric("Фейковые", fake_count)        # Таблица с цветами    styled = df.style.applymap(color_verdict, subset=["verdict"])    st.dataframe(styled)        # Детальный просмотр статьи    selected = st.selectbox("Выберите статью:", df["title"])    # 3 вкладки: Обзор, Анализ, Текст

**Streamlit** превращает Python-скрипт в веб-страницу.  Запуск: `streamlit run app.py`**Важные детали:**1. `st.session_state` — данные, которые сохраняются между нажатиями кнопок.2. `st.sidebar` — боковая панель для настроек.3. `st.checkbox(...)` — выбор источников (все включены по умолчанию).4. `color_verdict()` — функция, возвращающая CSS-стиль (зелёный/жёлтый/красный).---## 17. requirements.txt — зависимости

In [ ]:
# Science Paper Analyzer — зависимостиstreamlit>=1.28.0       # веб-интерфейсrequests>=2.28.0        # HTTP-запросы к APIpandas>=1.5.0           # таблицы и DataFrameopenpyxl>=3.0.0         # Excel с форматированиемlxml>=4.9.0             # быстрый парсер XML/HTMLbeautifulsoup4>=4.11.0  # парсинг HTML (CyberLeninka, JMLR)numpy>=1.23.0           # численные расчётыscikit-learn>=1.2.0     # ML-модели (запас)plotly>=5.14.0          # графики (запас)

**Зачем каждый:**- `requests` — все 10 источников опрашиваются через `requests.get()`.- `pandas` — результаты возвращаются как DataFrame.- `openpyxl` — для .xlsx с цветами.- `lxml` + `beautifulsoup4` — парсят HTML CyberLeninka, JMLR и XML arXiv.- `scikit-learn` и `plotly` — запасные (для будущих ML-моделей и графиков).---## 🎓 Итог: как всё работает### Поток данных

```Вы вводите тему → PaperAnalyzer.collect_papers()                        │           ┌────────────┼────────────┐           ▼            ▼            ▼       arXiv API    CrossRef API   CyberLeninka  ← 10 парсеров           │            │            │           └────────────┼────────────┘                        ▼              Список словарей (статьи)                        │                        ▼              PaperAnalyzer.analyze_papers()                        │           ┌────────────┼────────────┐           ▼            ▼            ▼    Citation        Text         Journal       ← 3 анализатора    Analyzer       Analyzer      Analyzer           │            │            │           └────────────┼────────────┘                        ▼              pandas DataFrame с вердиктами                        │                        ▼              CSV / Excel со скачиванием```

### Интерпретация результатов| Вердикт | Score | Что значит ||---------|-------|------------|| **Real** ✅ | ≥ 0.70 | Статья цитируется, текст естественный, площадка доверенная || **Suspicious** ⚠️ | 0.40–0.70 | Что-то не так — проверьте детали || **Fake** ❌ | < 0.40 | Серьёзные проблемы: хищнический журнал, AI-текст, нет цитирований |### Быстрые ссылки- **Запустить анализ:** https://colab.research.google.com/github/DmitPerson42/science-paper-analyzer/blob/main/science_paper_analyzer.ipynb- **Репозиторий:** https://github.com/DmitPerson42/science-paper-analyzer- **Тема диссертации:** Разработка методов фильтрации генеративных текстовых данных для предотвращения коллапса языковых моделей